# The goal of this notebook is to extract all the columns of the MATCH-V5 API and get them into the normalized database as your single source of truth, then do what ever u want with the data from there
## Sub Goals:
        1) The goal is to use the Merged_Player_IDs_{...}.csv and extract Player IDs from it. Where we will need to extract 1 ID per tier and rank, then extract 50 match_ids.
        2) If that player does not have enought matches we will have to the next player in the same rank

## Result:
        1) lower_rank_player_ids_collected = (7 Ranks) * (4 divisions) * (50 solo queue matches) 
                == 40,000 matches across all tiers
                == 4,000 matches for each tier

# Import

In [1]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time
from datetime import datetime

load_dotenv()
api_key = os.environ.get('riot_api_key')

# Saves Data to a CSV file locally

In [11]:
def save_match_data_to_csv(match_data_dict, base_path=None):
    """Save match data to CSV files per division."""
    
    if base_path is None:
        current_dir = os.getcwd()
        base_path = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'match data'))
    
    os.makedirs(base_path, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    for division, matches in match_data_dict.items():
        if not matches:
            continue
        
        rows = []
        for m in matches:
            match_info = m['match_data'].get('info', {})
            rows.append({
                'tier': m['tier'],
                'rank': m['rank'],
                'match_id': m['match_id'],
                'queueType': m['queueType'],
                'region': m['region'],
                'sub_region': m['sub_region'],
                'gameCreation': match_info.get('gameCreation'),
                'gameDuration': match_info.get('gameDuration'),
                'gameVersion': match_info.get('gameVersion'),
                'match_data': m['match_data']
            })
        
        df = pd.DataFrame(rows)
        filename = f"{division}_match_data_{timestamp}.csv"
        filepath = os.path.join(base_path, filename)
        df.to_csv(filepath, index=False)
        print(f"Saved {len(df)} matches to: {filepath}")


# The function that retrieves Match Data

In [3]:
def get_match_data_from_match_id(region='europe',sub_region='eun1', matchId=None):
    '''Retrieve detailed information about a specific match.

    Fetches comprehensive match data including participant statistics, team 
    compositions, game timeline, and results for a single match. This is 
    the second step in the match data retrieval process.

    The returned data is a large JSON object containing:
    - Match metadata (game version, duration, creation timestamp, queue type)
    - Team-level data (bans, objectives, win status)
    - Player-level data (champion played, KDA, damage, gold, items, runes, etc.)

    Args:
        region (str, optional): The Riot API routing region where the match 
            was played. Must match the region prefix in the matchId.
            Available values:
                - 'americas'  (NA, BR, LAN, LAS)
                - 'europe'    (EUW, EUNE, TR, RU)
                - 'asia'      (KR, JP)
                - 'sea'       (OCE, SG, PH, TW, VN, TH)
            Defaults to 'europe'.
        matchId (str, optional): The unique match identifier string.
            Format: "{platform}_{numeric_id}" (e.g., "EUN1_1234567890").
            Obtained from get_match_history_ids() function.
            Defaults to None.
            Example: "EUN1_1234567890"

    Returns:
        dict: A large JSON object containing complete match data. Key fields include:
            - metadata: matchId, participant PUUIDs, game version
            - info.gameCreation: Timestamp when the game was created
            - info.gameDuration: Length of game in seconds
            - info.queueId: Type of queue (420 = ranked solo, 440 = flex, etc.)
            - info.teams[2]: Team-level data (bans, objectives, win/loss)
            - info.participants[10]: Array of player data including:
                - summonerName, puuid, championName
                - kills, deaths, assists, KDA
                - totalDamageDealtToChampions, goldEarned
                - items, summonerSpells, runes
                - visionScore, wardsPlaced

    Raises:
        requests.exceptions.RequestException: If the API request fails.
        KeyError: If the matchId doesn't exist or the API response structure changes.

    Note:
        - Requires a globally defined `api_key` variable for authentication.
        - Each match detail request counts against your rate limit.
        - Match data can be very large (100KB+ per match) - consider caching.
        - The region parameter MUST match the region where the match was played.
        - Some match data fields may be null for very old matches or certain game modes.

    Example:
        # Get a single match's details
        match_ids = get_match_history_ids(puuid="abc123...", count=1)
        match_data = get_match_data_from_match_id(matchId=match_ids[0])
        
        # Extract player stats
        player = match_data['info']['participants'][0]
        print(f"KDA: {player['kills']}/{player['deaths']}/{player['assists']}")
        print(f"Champion: {player['championName']}")
    '''

    root_url = f'https://{region}.api.riotgames.com/'
    endpoint = f'lol/match/v5/matches/{matchId}'

    response = requests.get(root_url + endpoint + '?api_key=' + api_key)

    return response.json()

In [13]:
class RateLimiter:
    def __init__(self):
        self.short_term_requests = []
        self.long_term_requests = []
        self.short_window = 1
        self.long_window = 120
        self.short_limit = 20
        self.long_limit = 100
    
    def wait_if_needed(self):
        current_time = time.time()
        self.short_term_requests = [t for t in self.short_term_requests if current_time - t < self.short_window]
        self.long_term_requests = [t for t in self.long_term_requests if current_time - t < self.long_window]
        
        if len(self.short_term_requests) >= self.short_limit:
            sleep_time = self.short_term_requests[0] + self.short_window - current_time
            if sleep_time > 0:
                print(f"Short-term rate limit reached. Waiting {sleep_time:.2f}s...")
                time.sleep(sleep_time)
                return self.wait_if_needed()
        
        if len(self.long_term_requests) >= self.long_limit:
            sleep_time = self.long_term_requests[0] + self.long_window - current_time
            if sleep_time > 0:
                print(f"Long-term rate limit reached. Waiting {sleep_time:.2f}s...")
                time.sleep(sleep_time)
                return self.wait_if_needed()
    
    def add_request(self):
        current_time = time.time()
        self.short_term_requests.append(current_time)
        self.long_term_requests.append(current_time)

In [ ]:
def collect_match_data_from_merged_file(region='europe', sub_region='eun1', matches_per_division=None):
    """Read merged match IDs file and fetch full match data for each match ID.
    
    Args:
        api_key: Riot API key
        region: API region routing
        sub_region: Platform sub-region
        matches_per_division: dict or int. Number of matches to process per division.
                             Default: 50 for Iron-Diamond, 200 for Master+.
    """
    
    if matches_per_division is None:
        matches_per_division = {
            'IRON': 50, 'BRONZE': 50, 'SILVER': 50, 'GOLD': 50,
            'PLATINUM': 50, 'EMERALD': 50, 'DIAMOND': 50,
            'MASTER': 200, 'GRANDMASTER': 200, 'CHALLENGER': 200
        }
    elif isinstance(matches_per_division, int):
        matches_per_division = {tier: matches_per_division for tier in 
                               ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND', 'MASTER', 'GRANDMASTER', 'CHALLENGER']}
    
    limiter = RateLimiter()
    
    # Load merged match IDs
    current_dir = os.getcwd()
    data_path = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'match ids'))
    
    all_files = [f for f in os.listdir(data_path) if f.startswith('Merged_Match_IDs_') and f.endswith('.csv')]
    if not all_files:
        print("No merged match IDs file found")
        return {}
    
    latest_file = sorted(all_files, reverse=True)[0]
    df = pd.read_csv(os.path.join(data_path, latest_file))
    print(f"Loaded {len(df)} match IDs from {latest_file}")
    
    # Get unique divisions
    divisions = df['division'].unique()
    
    results = {}
    
    for division in divisions:
        tier = division.split('_')[0]
        target = matches_per_division.get(tier, 50)
        
        div_df = df[df['division'] == division].head(target)
        
        if div_df.empty:
            print(f"\n{division}: No match IDs found")
            continue
        
        print(f"\n{division} (fetching {len(div_df)} matches):")
        match_data_list = []
        
        for idx, row in div_df.iterrows():
            match_id = row['match_id']
            
            try:
                limiter.wait_if_needed()
                match_data = get_match_data_from_match_id(region=region, sub_region=sub_region, matchId=match_id)
                limiter.add_request()
                
                match_data_list.append({
                    'tier': row['tier'],
                    'rank': row['rank'],
                    'match_id': match_id,
                    'queueType': row['queueType'],
                    'region': row['region'],
                    'sub_region': row['sub_region'],
                    'match_data': match_data
                })
                
                if (len(match_data_list)) % 10 == 0:
                    print(f"  Fetched {len(match_data_list)}/{len(div_df)} matches")
                
            except Exception as e:
                print(f"  Error fetching {match_id}: {e}")
                time.sleep(5)
                continue
        
        results[division] = match_data_list
        print(f"  Final: {len(results[division])} matches fetched")
    
    return results

In [15]:
match_data = collect_match_data_from_merged_file( region='europe', sub_region='eun1')

Loaded 2000 match IDs from Merged_Match_IDs_20260613_160406.csv

IRON_I (fetching 50 matches):
  Fetched 10/50 matches
  Fetched 20/50 matches
  Fetched 30/50 matches
  Fetched 40/50 matches
  Fetched 50/50 matches
  Final: 50 matches fetched

IRON_II (fetching 50 matches):
  Fetched 10/50 matches
  Fetched 20/50 matches
  Fetched 30/50 matches
  Fetched 40/50 matches
  Fetched 50/50 matches
  Final: 50 matches fetched

IRON_III (fetching 50 matches):
  Fetched 10/50 matches
  Fetched 20/50 matches
  Fetched 30/50 matches
  Fetched 40/50 matches
  Fetched 50/50 matches
  Final: 50 matches fetched

IRON_IV (fetching 50 matches):
  Fetched 10/50 matches
  Fetched 20/50 matches
  Fetched 30/50 matches
  Fetched 40/50 matches
  Fetched 50/50 matches
  Final: 50 matches fetched

BRONZE_I (fetching 50 matches):
  Fetched 10/50 matches
  Fetched 20/50 matches
  Fetched 30/50 matches
  Fetched 40/50 matches
  Fetched 50/50 matches
  Final: 50 matches fetched

BRONZE_II (fetching 50 matches):
 

In [3]:
temp_df= get_match_data_from_match_id(matchId='EUN1_3964311443') 

In [4]:
temp_df

{'metadata': {'dataVersion': '2',
  'matchId': 'EUN1_3964311443',
  'participants': ['KSzaEfHnjF_bQQR5fS0Q0V6o0xwc6g7aQ2tL0fE9ZLbE6OkHI1ZjelghIDvVtj-_LUTgkMBdGJmmJg',
   'hdhANkNP8nMCbpNDkLfEHJ_bXw10gbiGU6d2Hs_iKTQ5EyCOUvOqufbktkC-GKRxmzV0Hz5vh4DroA',
   'hfMP5_VTMFoT1hzMv2U_yBDeFOxz0BTtI4EUDhMbUu54WI67gItXOWsf79vt3B6HHvPDxgA3wdWONQ',
   'dNLwsXCOAlBqZGBN6FHtKQtEvDCbOnyqVkM0zjle7vivEvgpzzkLXyfsuHZBZolgcfoAcnHTZtLhrg',
   'SAOb3wZRVjiihf6WVCrZSrCK9WaUeR9bR6sh8trC2yvDJwI6cA4UAnRUnTfS9IRiDyAtKFfv8mwU-g',
   'ggJ93BTVO4GbUuEtNkG3OpY8fMIU2btPT_UrHMjcRqoKogTVlhBD4Gw9xWcw3P3Mxqc9Nok3jxNX3A',
   'Stjrfi4OWd2tY0f60N_YaPpbaH_1C3NGqVSRZW8-ovF8F6-cgyjmfF6bCIs39MmvPOLIWcW-ymKcaw',
   'Ry-KeWegorCK2ejhu4XBzp_2NiC0pL-XxzcQhTtXcRr1j6wbJV9ZMwGtEWpBKRfAdv9GE_5aIQRqRw',
   'pIh5eyz4uUdlc9w4lCWYH2mvxnZ_t43q0ncXjwbCpthVMgbnJz9uLZLMjATOvnjN69gQKzra6EQ1gQ',
   'CQPvgtpRFkmiHHaHTwU16TtWE4A4is925HN_VcKNVYli5isypZaVdbCoXIlF4Y8Sux-TUwnUCwTvXQ']},
 'info': {'endOfGameResult': 'GameComplete',
  'gameCreation': 178

# Team Data Only Code (WORKING)

In [ ]:
# ChallengesDto is in participants['challenges'] (BIG BOIIII)
# MissionsDto	is in participants['missions'] (Useless: I think used to track the battle pass)
# PerksDto is in participants['perks'] (DONE)
# PerkStatsDto is in participants['perks']['statPerks'] (DONE)
# PerkStyleDto is in participants['perks']['styles'] (DONE)
# BanDto is in teams['bans'] (DONE)
# ObjectivesDto is in teams['objectives'] (DONE) This contains the objectives taken on the whole team grain


# The JSON Response

game = temp_df



# Main Data Frames that contains all the sub lists

metadata = game['metadata']
info = game['info']



# Getting what is useful from metadata dto

# dataVersion is not useful for us, i think it is related to riot data cleaning phases or something similar
matchid = metadata['matchId'] # MatchId: The Unique Identifier for the match
# No need to get the participants puuids list from here, we will get it from info['participants']['puuid']



# Getting the useful dataout of the info data frame
gameDuration = info['gameDuration'] # Measured in Seconds (Need a cleaning/mathematical function to make it look like MM:SS)
platformId = info['platformId'] # Refers to the sub-region this game is taking place at
queueId = info['queueId'] # Refers to the queueId, e.g., 420: SOLO QUEUE 5*5



# Teams grain and their objectives

teams = info['teams']
team_rows = []  # Store flattened team data

for team in teams:

    teamid = team['teamId']
    win = team['win']

    # ============================================
    # Flatten Bans
    # ============================================
    bans = team['bans']
    bans_df = pd.DataFrame(bans)  # Convert list of dicts to DataFrame
    
    # Create ban columns with champion IDs
    ban_dict = {}
    for i, row in bans_df.iterrows():
        ban_dict[f'ban_{i+1}_championId'] = row['championId']
        # ban_dict[f'ban_{i+1}_pickTurn'] = row['pickTurn'] # Not useful for me
    
    # ============================================
    # Flatten Objectives
    # ============================================
    obj = team['objectives']
    
    # Create objective columns: first and kills for each objective type
    obj_dict = {}
    for objective_name, objective_data in obj.items():
        obj_dict[f'obj_{objective_name}_first'] = objective_data['first']
        obj_dict[f'obj_{objective_name}_kills'] = objective_data['kills']
    
    # ============================================
    # Combine Everything
    # ============================================
    team_row = {
        'matchId': matchid,
        'teamId': teamid,
        'win': win
    }
    team_row.update(ban_dict)
    team_row.update(obj_dict)
    
    team_rows.append(team_row)

# # Create final DataFrame
# teams_df = pd.DataFrame(team_rows)

# teams_df









,matchId,teamId,win,ban_1_championId,ban_1_pickTurn,ban_2_championId,ban_2_pickTurn,ban_3_championId,ban_3_pickTurn,ban_4_championId,...,obj_dragon_first,obj_dragon_kills,obj_horde_first,obj_horde_kills,obj_inhibitor_first,obj_inhibitor_kills,obj_riftHerald_first,obj_riftHerald_kills,obj_tower_first,obj_tower_kills
0,EUN1_3964311443,100,False,39,1,111,2,25,3,238,...,False,1,False,0,False,0,False,0,False,2
1,EUN1_3964311443,200,True,72,6,119,7,902,8,200,...,True,4,True,3,True,3,True,1,True,11


# Match Participants  + Challenges + Perks Data (WORKING)

In [ ]:
import pandas as pd


# ChallengeDto Related Variables:
# Nore we will include the challenges for 1 reason: they might be useful for feature engineering later on (although it contains lots of NULLs/NaNs)
# Fields to EXCLUDE (non-Summoner's Rift modes)
EXCLUDE_CHALLENGES = {
    # ARAM-specific
    'killsOnRecentlyHealedByAramPack',
    'poroExplosions',
    'snowballsHit',
    'abilityUses',  # ARAM-specific challenge
    
    # Arena-specific
    'fistBumpParticipation',
    
    # Swarm / PvE mode
    'SWARM_DefeatAatrox',
    'SWARM_DefeatBriar',
    'SWARM_DefeatMiniBosses',
    'SWARM_EvolveWeapon',
    'SWARM_Have3Passives',
    'SWARM_KillEnemy',
    'SWARM_PickupGold',
    'SWARM_ReachLevel50',
    'SWARM_Survive15Min',
    'SWARM_WinWith5EvolvedWeapons',
    
    # Unknown/Other modes
    'dancedWithRiftHerald',  # Special event
    'legendaryItemUsed',  # List[int] — different data type, handle separately if needed
    'InfernalScalePickup',  # Arena/event specific
}

# Fields that are List type (not int/float/bool) — handle separately (WORK ON THEM LATER IF NEEDED - NOT NOW)
# LIST_TYPE_CHALLENGES = {
#     'legendaryItemUsed',
# }


players = info['participants']
player_rows = []

for player in players:
    # ============================================
    # Core Identity Fields
    # ============================================
    row = {
        'matchId': matchid,
        'teamId':player.get('teamId'),
        'participantId': player.get('participantId'),
        'puuid': player.get('puuid'),
        #'summonerId': player.get('summonerId'),
        #'summonerName': player.get('summonerName'),
        'riotIdGameName': player.get('riotIdGameName'),
        'riotIdTagline': player.get('riotIdTagline'),
        'profileIcon': player.get('profileIcon'),
        'summonerLevel': player.get('summonerLevel'),
        
        # ============================================
        # Team & Position
        # ============================================

        'teamPosition': player.get('teamPosition'), 
        # 'role': player.get('role'), # Legacy and not reliable
        # 'lane': player.get('lane'), # Legacy and not reliable
        # 'playerSubteamId': player.get('playerSubteamId'), # Useless for me
        # 'subteamPlacement': player.get('subteamPlacement'), # Useless for me
        # 'placement': player.get('placement'), # Useless for me
        # 'individualPosition': player.get('individualPosition'), # Redundant
        # 'riotIdGameName': player.get('teamId'), # Redundant from above 


        # ============================================
        # Champion Info
        # ============================================
        'championId': player.get('championId'),
        'championName': player.get('championName'),
        'champLevel': player.get('champLevel'),
        # 'champExperience': player.get('champExperience'),
        # 'championTransform': player.get('championTransform'),
        
        # ============================================
        # Game Outcome
        # ============================================
        # 'win': player.get('win'), # Redundant
        # 'gameEndedInEarlySurrender': player.get('gameEndedInEarlySurrender'),
        # 'gameEndedInSurrender': player.get('gameEndedInSurrender'),
        # 'teamEarlySurrendered': player.get('teamEarlySurrendered'),
        # 'eligibleForProgression': player.get('eligibleForProgression'),
        
        # ============================================
        # KDA
        # ============================================
        'kills': player.get('kills'),
        'deaths': player.get('deaths'),
        'assists': player.get('assists'),
        
        # ============================================
        # Multi-Kills
        # ============================================
        'doubleKills': player.get('doubleKills'),
        'tripleKills': player.get('tripleKills'),
        'quadraKills': player.get('quadraKills'),
        'pentaKills': player.get('pentaKills'),
        # 'unrealKills': player.get('unrealKills'),
        'largestMultiKill': player.get('largestMultiKill'),
        'largestKillingSpree': player.get('largestKillingSpree'),
        'killingSprees': player.get('killingSprees'),
        
        # ============================================
        # First Blood / Tower
        # ============================================
        'firstBloodKill': player.get('firstBloodKill'),
        'firstBloodAssist': player.get('firstBloodAssist'),
        'firstTowerKill': player.get('firstTowerKill'),
        'firstTowerAssist': player.get('firstTowerAssist'),
        
        # ============================================
        # Damage Dealt
        # ============================================
        'totalDamageDealt': player.get('totalDamageDealt'),
        'totalDamageDealtToChampions': player.get('totalDamageDealtToChampions'),
        'physicalDamageDealt': player.get('physicalDamageDealt'),
        'physicalDamageDealtToChampions': player.get('physicalDamageDealtToChampions'),
        'magicDamageDealt': player.get('magicDamageDealt'),
        'magicDamageDealtToChampions': player.get('magicDamageDealtToChampions'),
        'trueDamageDealt': player.get('trueDamageDealt'),
        'trueDamageDealtToChampions': player.get('trueDamageDealtToChampions'),
        'damageDealtToBuildings': player.get('damageDealtToBuildings'),
        'damageDealtToObjectives': player.get('damageDealtToObjectives'),
        'damageDealtToTurrets': player.get('damageDealtToTurrets'),
        'largestCriticalStrike': player.get('largestCriticalStrike'),
        
        # ============================================
        # Damage Taken & Mitigated
        # ============================================
        'totalDamageTaken': player.get('totalDamageTaken'),
        'physicalDamageTaken': player.get('physicalDamageTaken'),
        'magicDamageTaken': player.get('magicDamageTaken'),
        'trueDamageTaken': player.get('trueDamageTaken'),
        'damageSelfMitigated': player.get('damageSelfMitigated'),
        
        # ============================================
        # Healing & Shielding
        # ============================================
        'totalHeal': player.get('totalHeal'),
        'totalHealsOnTeammates': player.get('totalHealsOnTeammates'),
        'totalUnitsHealed': player.get('totalUnitsHealed'),
        'totalDamageShieldedOnTeammates': player.get('totalDamageShieldedOnTeammates'),
        
        # ============================================
        # Gold & Economy
        # ============================================
        'goldEarned': player.get('goldEarned'),
        'goldSpent': player.get('goldSpent'),
        'consumablesPurchased': player.get('consumablesPurchased'),
        'itemsPurchased': player.get('itemsPurchased'),
        'bountyLevel': player.get('bountyLevel'),
        
        # ============================================
        # Items (0-6 slots + trinket in slot 6)
        # ============================================
        'item0': player.get('item0'),
        'item1': player.get('item1'),
        'item2': player.get('item2'),
        'item3': player.get('item3'),
        'item4': player.get('item4'),
        'item5': player.get('item5'),
        'item6': player.get('item6'),
        
        # ============================================
        # Summoner Spells
        # ============================================
        'summoner1Id': player.get('summoner1Id'),
        'summoner1Casts': player.get('summoner1Casts'),
        'summoner2Id': player.get('summoner2Id'),
        'summoner2Casts': player.get('summoner2Casts'),
        
        # ============================================
        # Ability Casts
        # ============================================
        'spell1Casts': player.get('spell1Casts'),
        'spell2Casts': player.get('spell2Casts'),
        'spell3Casts': player.get('spell3Casts'),
        'spell4Casts': player.get('spell4Casts'),
        
        # ============================================
        # Minions & Jungle
        # ============================================
        'totalMinionsKilled': player.get('totalMinionsKilled'),
        'neutralMinionsKilled': player.get('neutralMinionsKilled'),
        'totalAllyJungleMinionsKilled': player.get('totalAllyJungleMinionsKilled'),
        'totalEnemyJungleMinionsKilled': player.get('totalEnemyJungleMinionsKilled'),
        
        # ============================================
        # Objectives
        # ============================================
        'baronKills': player.get('baronKills'),
        'dragonKills': player.get('dragonKills'),
        'inhibitorKills': player.get('inhibitorKills'),
        'inhibitorTakedowns': player.get('inhibitorTakedowns'),
        'inhibitorsLost': player.get('inhibitorsLost'),
        'turretKills': player.get('turretKills'),
        'turretTakedowns': player.get('turretTakedowns'),
        'turretsLost': player.get('turretsLost'),
        'nexusKills': player.get('nexusKills'),
        'nexusTakedowns': player.get('nexusTakedowns'),
        'nexusLost': player.get('nexusLost'),
        'objectivesStolen': player.get('objectivesStolen'),
        'objectivesStolenAssists': player.get('objectivesStolenAssists'),
        
        # ============================================
        # Vision
        # ============================================
        'visionScore': player.get('visionScore'),
        'wardsPlaced': player.get('wardsPlaced'),
        'wardsKilled': player.get('wardsKilled'),
        'sightWardsBoughtInGame': player.get('sightWardsBoughtInGame'),
        'visionWardsBoughtInGame': player.get('visionWardsBoughtInGame'),
        'detectorWardsPlaced': player.get('detectorWardsPlaced'),
        
        # ============================================
        # CC (Crowd Control)
        # ============================================
        'timeCCingOthers': player.get('timeCCingOthers'),
        'totalTimeCCDealt': player.get('totalTimeCCDealt'),
        
        # ============================================
        # Time
        # ============================================
        'timePlayed': player.get('timePlayed'),
        'totalTimeSpentDead': player.get('totalTimeSpentDead'),
        'longestTimeSpentLiving': player.get('longestTimeSpentLiving'),
        
        # ============================================
        # Pings (Communication)
        # ============================================
        'allInPings': player.get('allInPings'),
        'assistMePings': player.get('assistMePings'),
        'commandPings': player.get('commandPings'),
        'enemyMissingPings': player.get('enemyMissingPings'),
        'enemyVisionPings': player.get('enemyVisionPings'),
        'holdPings': player.get('holdPings'),
        'getBackPings': player.get('getBackPings'),
        'needVisionPings': player.get('needVisionPings'),
        'onMyWayPings': player.get('onMyWayPings'),
        'pushPings': player.get('pushPings'),
        'visionClearedPings': player.get('visionClearedPings'),
        
        # ============================================
        # Player Scores (0-11) (NOT UNDERSTOOD)
        # ============================================
        # 'playerScore0': player.get('playerScore0'),
        # 'playerScore1': player.get('playerScore1'),
        # 'playerScore2': player.get('playerScore2'),
        # 'playerScore3': player.get('playerScore3'),
        # 'playerScore4': player.get('playerScore4'),
        # 'playerScore5': player.get('playerScore5'),
        # 'playerScore6': player.get('playerScore6'),
        # 'playerScore7': player.get('playerScore7'),
        # 'playerScore8': player.get('playerScore8'),
        # 'playerScore9': player.get('playerScore9'),
        # 'playerScore10': player.get('playerScore10'),
        # 'playerScore11': player.get('playerScore11'),
        
        # ============================================
        # Augments (Arena/Game Mode specific) (NOT APPLICABLE FOR SOLO QUEUE)
        # ============================================
        # 'playerAugment1': player.get('playerAugment1'),
        # 'playerAugment2': player.get('playerAugment2'),
        # 'playerAugment3': player.get('playerAugment3'),
        # 'playerAugment4': player.get('playerAugment4'),
    }

    # ============================================
    # Flatten Perks (Runes)
    # ============================================
    perks = player.get('perks', {})
    if perks:
        # ============================================
        # Stat Shards (statPerks)
        # ============================================
        stat_perks = perks.get('statPerks', {})
        if stat_perks:
            row['perk_stat_defense'] = stat_perks.get('defense')
            row['perk_stat_flex'] = stat_perks.get('flex')
            row['perk_stat_offense'] = stat_perks.get('offense')
        
        # ============================================
        # Rune Styles (Primary & Secondary Trees + Selections)
        # ============================================
        styles = perks.get('styles', [])  # List of PerkStyleDto
        for style_idx, style in enumerate(styles):
            # Determine if primary or secondary based on list position
            # First element = Primary, Second element = Secondary
            if style_idx == 0:
                prefix = 'perk_primary'
            elif style_idx == 1:
                prefix = 'perk_secondary'
            else:
                prefix = f'perk_extra_{style_idx}'  # In case there are more than 2
            
            # The style (rune tree) ID — THIS is the primary/secondary style
            row[f'{prefix}_style'] = style.get('style')
            row[f'{prefix}_style_description'] = style.get('description', '')
            
            # Individual rune selections within this style
            selections = style.get('selections', [])
            for sel_idx, selection in enumerate(selections):
                rune_num = sel_idx + 1
                row[f'{prefix}_rune_{rune_num}'] = selection.get('perk')
                row[f'{prefix}_rune_{rune_num}_var1'] = selection.get('var1')
                row[f'{prefix}_rune_{rune_num}_var2'] = selection.get('var2')
                row[f'{prefix}_rune_{rune_num}_var3'] = selection.get('var3')

                
    # ============================================
    # Flatten Challenges (Solo Queue / Summoner's Rift only)
    # ============================================
    challenges = player.get('challenges', {})
    if challenges:
        for challenge_key, challenge_value in challenges.items():
            if challenge_key in EXCLUDE_CHALLENGES:
                continue
            # if challenge_key in LIST_TYPE_CHALLENGES:
            #     continue
            row[f'challenge_{challenge_key}'] = challenge_value


    player_rows.append(row)

# Create final DataFrame
players_df = pd.DataFrame(player_rows)
players_df



,matchId,teamId,participantId,puuid,riotIdGameName,riotIdTagline,profileIcon,summonerLevel,teamPosition,championId,...,challenge_wardsGuarded,challenge_junglerKillsEarlyJungle,challenge_killsOnLanersEarlyJungleAsJungler,challenge_earliestBaron,challenge_firstTurretKilledTime,challenge_highestChampionDamage,challenge_highestCrowdControlScore,challenge_soloTurretsLategame,challenge_teleportTakedowns,challenge_highestWardKills
0,EUN1_3964311443,100,1,KSzaEfHnjF_bQQR5fS0Q0V6o0xwc6g7aQ2tL0fE9ZLbE6O...,HSYXA GTPS,EUNE,5269,921,TOP,58,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,EUN1_3964311443,100,2,hdhANkNP8nMCbpNDkLfEHJ_bXw10gbiGU6d2Hs_iKTQ5Ey...,Ø AMATERASU Ø,002,5935,759,JUNGLE,131,...,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,EUN1_3964311443,100,3,hfMP5_VTMFoT1hzMv2U_yBDeFOxz0BTtI4EUDhMbUu54WI...,xShadowsx12,EUNE,657,874,MIDDLE,10,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,EUN1_3964311443,100,4,dNLwsXCOAlBqZGBN6FHtKQtEvDCbOnyqVkM0zjle7vivEv...,Blackliht,afek,7048,695,BOTTOM,235,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,EUN1_3964311443,100,5,SAOb3wZRVjiihf6WVCrZSrCK9WaUeR9bR6sh8trC2yvDJw...,LPL Aggression,EUNE,691,493,UTILITY,432,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,EUN1_3964311443,200,6,ggJ93BTVO4GbUuEtNkG3OpY8fMIU2btPT_UrHMjcRqoKog...,NaDnoZNimi,naut,3221,613,TOP,17,...,0,NaN,NaN,1240.494848,813.779288,1.0,1.0,1.0,1.0,NaN
6,EUN1_3964311443,200,7,Stjrfi4OWd2tY0f60N_YaPpbaH_1C3NGqVSRZW8-ovF8F6...,Ljubišguze,pice,7126,77,JUNGLE,11,...,0,0.0,0.0,1240.494848,813.779288,NaN,NaN,2.0,NaN,NaN
7,EUN1_3964311443,200,8,Ry-KeWegorCK2ejhu4XBzp_2NiC0pL-XxzcQhTtXcRr1j6...,CLIMBING,funny,7102,1600,MIDDLE,134,...,0,NaN,NaN,1240.494848,813.779288,NaN,NaN,1.0,NaN,NaN
8,EUN1_3964311443,200,9,pIh5eyz4uUdlc9w4lCWYH2mvxnZ_t43q0ncXjwbCpthVMg...,PogromcaCzekolad,GODS,7070,1232,BOTTOM,202,...,0,NaN,NaN,1240.494848,813.779288,NaN,NaN,1.0,NaN,NaN
9,EUN1_3964311443,200,10,CQPvgtpRFkmiHHaHTwU16TtWE4A4is925HN_VcKNVYli5i...,Taplia only 1,Oswl,6934,895,UTILITY,517,...,0,NaN,NaN,1240.494848,813.779288,NaN,NaN,NaN,NaN,1.0


# NO RETRY LOGIC HERE (FOR FAILED MATCHES)
# Reads match ids and extracts match data and saves it to a csv locally

In [ ]:
class RateLimiter:
    def __init__(self):
        self.short_term_requests = []
        self.long_term_requests = []
        self.short_window = 1
        self.long_window = 120
        self.short_limit = 20
        self.long_limit = 100
    
    def wait_if_needed(self):
        current_time = time.time()
        self.short_term_requests = [t for t in self.short_term_requests if current_time - t < self.short_window]
        self.long_term_requests = [t for t in self.long_term_requests if current_time - t < self.long_window]
        
        if len(self.short_term_requests) >= self.short_limit:
            sleep_time = self.short_term_requests[0] + self.short_window - current_time
            if sleep_time > 0:
                print(f"Short-term rate limit reached. Waiting {sleep_time:.2f}s...")
                time.sleep(sleep_time)
                return self.wait_if_needed()
        
        if len(self.long_term_requests) >= self.long_limit:
            sleep_time = self.long_term_requests[0] + self.long_window - current_time
            if sleep_time > 0:
                print(f"Long-term rate limit reached. Waiting {sleep_time:.2f}s...")
                time.sleep(sleep_time)
                return self.wait_if_needed()
    
    def add_request(self):
        current_time = time.time()
        self.short_term_requests.append(current_time)
        self.long_term_requests.append(current_time)


def get_match_data_from_match_id(region='europe', sub_region='eun1', matchId=None):
    '''Retrieve detailed information about a specific match.'''
    root_url = f'https://{region}.api.riotgames.com/'
    endpoint = f'lol/match/v5/matches/{matchId}'
    response = requests.get(root_url + endpoint + '?api_key=' + api_key)
    return response.json()


def flatten_team_data(match_data, match_id):
    """Flatten team-level data from match JSON."""
    teams = match_data['info']['teams']
    team_rows = []

    for team in teams:
        teamid = team['teamId']
        win = team['win']

        # Flatten Bans
        bans = team['bans']
        ban_dict = {}
        for i, ban in enumerate(bans):
            ban_dict[f'ban_{i+1}_championId'] = ban['championId']
        
        # Flatten Objectives
        obj = team['objectives']
        obj_dict = {}
        for objective_name, objective_data in obj.items():
            obj_dict[f'obj_{objective_name}_first'] = objective_data['first']
            obj_dict[f'obj_{objective_name}_kills'] = objective_data['kills']
        
        # Combine Everything
        team_row = {
            'matchId': match_id,
            'gameDuration': match_data['info']['gameDuration'],
            'platformId': match_data['info']['platformId'],
            'queueId': match_data['info']['queueId'],
            'teamId': teamid,
            'win': win
        }
        team_row.update(ban_dict)
        team_row.update(obj_dict)
        
        team_rows.append(team_row)
    
    return team_rows


def flatten_player_data(match_data, match_id):
    """Flatten player-level data from match JSON."""
    
    # ChallengeDto Related Variables - Fields to EXCLUDE (non-Summoner's Rift modes)
    EXCLUDE_CHALLENGES = {
        'killsOnRecentlyHealedByAramPack', 'poroExplosions', 'snowballsHit',
        'abilityUses', 'fistBumpParticipation',
        'SWARM_DefeatAatrox', 'SWARM_DefeatBriar', 'SWARM_DefeatMiniBosses',
        'SWARM_EvolveWeapon', 'SWARM_Have3Passives', 'SWARM_KillEnemy',
        'SWARM_PickupGold', 'SWARM_ReachLevel50', 'SWARM_Survive15Min',
        'SWARM_WinWith5EvolvedWeapons', 'dancedWithRiftHerald',
        'legendaryItemUsed', 'InfernalScalePickup',
    }
    
    players = match_data['info']['participants']
    player_rows = []

    for player in players:
        row = {
            # Core Identity Fields
            'matchId': match_id,
            'gameDuration': match_data['info']['gameDuration'],
            'platformId': match_data['info']['platformId'],
            'queueId': match_data['info']['queueId'],
            'teamId': player.get('teamId'),
            'participantId': player.get('participantId'),
            'puuid': player.get('puuid'),
            'riotIdGameName': player.get('riotIdGameName'),
            'riotIdTagline': player.get('riotIdTagline'),
            'profileIcon': player.get('profileIcon'),
            'summonerLevel': player.get('summonerLevel'),
            
            # Team & Position
            'teamPosition': player.get('teamPosition'),
            
            # Champion Info
            'championId': player.get('championId'),
            'championName': player.get('championName'),
            'champLevel': player.get('champLevel'),
            
            # KDA
            'kills': player.get('kills'),
            'deaths': player.get('deaths'),
            'assists': player.get('assists'),
            
            # Multi-Kills
            'doubleKills': player.get('doubleKills'),
            'tripleKills': player.get('tripleKills'),
            'quadraKills': player.get('quadraKills'),
            'pentaKills': player.get('pentaKills'),
            'largestMultiKill': player.get('largestMultiKill'),
            'largestKillingSpree': player.get('largestKillingSpree'),
            'killingSprees': player.get('killingSprees'),
            
            # First Blood / Tower
            'firstBloodKill': player.get('firstBloodKill'),
            'firstBloodAssist': player.get('firstBloodAssist'),
            'firstTowerKill': player.get('firstTowerKill'),
            'firstTowerAssist': player.get('firstTowerAssist'),
            
            # Damage Dealt
            'totalDamageDealt': player.get('totalDamageDealt'),
            'totalDamageDealtToChampions': player.get('totalDamageDealtToChampions'),
            'physicalDamageDealt': player.get('physicalDamageDealt'),
            'physicalDamageDealtToChampions': player.get('physicalDamageDealtToChampions'),
            'magicDamageDealt': player.get('magicDamageDealt'),
            'magicDamageDealtToChampions': player.get('magicDamageDealtToChampions'),
            'trueDamageDealt': player.get('trueDamageDealt'),
            'trueDamageDealtToChampions': player.get('trueDamageDealtToChampions'),
            'damageDealtToBuildings': player.get('damageDealtToBuildings'),
            'damageDealtToObjectives': player.get('damageDealtToObjectives'),
            'damageDealtToTurrets': player.get('damageDealtToTurrets'),
            'largestCriticalStrike': player.get('largestCriticalStrike'),
            
            # Damage Taken & Mitigated
            'totalDamageTaken': player.get('totalDamageTaken'),
            'physicalDamageTaken': player.get('physicalDamageTaken'),
            'magicDamageTaken': player.get('magicDamageTaken'),
            'trueDamageTaken': player.get('trueDamageTaken'),
            'damageSelfMitigated': player.get('damageSelfMitigated'),
            
            # Healing & Shielding
            'totalHeal': player.get('totalHeal'),
            'totalHealsOnTeammates': player.get('totalHealsOnTeammates'),
            'totalUnitsHealed': player.get('totalUnitsHealed'),
            'totalDamageShieldedOnTeammates': player.get('totalDamageShieldedOnTeammates'),
            
            # Gold & Economy
            'goldEarned': player.get('goldEarned'),
            'goldSpent': player.get('goldSpent'),
            'consumablesPurchased': player.get('consumablesPurchased'),
            'itemsPurchased': player.get('itemsPurchased'),
            'bountyLevel': player.get('bountyLevel'),
            
            # Items (0-6 slots)
            'item0': player.get('item0'),
            'item1': player.get('item1'),
            'item2': player.get('item2'),
            'item3': player.get('item3'),
            'item4': player.get('item4'),
            'item5': player.get('item5'),
            'item6': player.get('item6'),
            
            # Summoner Spells
            'summoner1Id': player.get('summoner1Id'),
            'summoner1Casts': player.get('summoner1Casts'),
            'summoner2Id': player.get('summoner2Id'),
            'summoner2Casts': player.get('summoner2Casts'),
            
            # Ability Casts
            'spell1Casts': player.get('spell1Casts'),
            'spell2Casts': player.get('spell2Casts'),
            'spell3Casts': player.get('spell3Casts'),
            'spell4Casts': player.get('spell4Casts'),
            
            # Minions & Jungle
            'totalMinionsKilled': player.get('totalMinionsKilled'),
            'neutralMinionsKilled': player.get('neutralMinionsKilled'),
            'totalAllyJungleMinionsKilled': player.get('totalAllyJungleMinionsKilled'),
            'totalEnemyJungleMinionsKilled': player.get('totalEnemyJungleMinionsKilled'),
            
            # Objectives
            'baronKills': player.get('baronKills'),
            'dragonKills': player.get('dragonKills'),
            'inhibitorKills': player.get('inhibitorKills'),
            'inhibitorTakedowns': player.get('inhibitorTakedowns'),
            'inhibitorsLost': player.get('inhibitorsLost'),
            'turretKills': player.get('turretKills'),
            'turretTakedowns': player.get('turretTakedowns'),
            'turretsLost': player.get('turretsLost'),
            'nexusKills': player.get('nexusKills'),
            'nexusTakedowns': player.get('nexusTakedowns'),
            'nexusLost': player.get('nexusLost'),
            'objectivesStolen': player.get('objectivesStolen'),
            'objectivesStolenAssists': player.get('objectivesStolenAssists'),
            
            # Vision
            'visionScore': player.get('visionScore'),
            'wardsPlaced': player.get('wardsPlaced'),
            'wardsKilled': player.get('wardsKilled'),
            'sightWardsBoughtInGame': player.get('sightWardsBoughtInGame'),
            'visionWardsBoughtInGame': player.get('visionWardsBoughtInGame'),
            'detectorWardsPlaced': player.get('detectorWardsPlaced'),
            
            # CC (Crowd Control)
            'timeCCingOthers': player.get('timeCCingOthers'),
            'totalTimeCCDealt': player.get('totalTimeCCDealt'),
            
            # Time
            'timePlayed': player.get('timePlayed'),
            'totalTimeSpentDead': player.get('totalTimeSpentDead'),
            'longestTimeSpentLiving': player.get('longestTimeSpentLiving'),
            
            # Pings (Communication)
            'allInPings': player.get('allInPings'),
            'assistMePings': player.get('assistMePings'),
            'commandPings': player.get('commandPings'),
            'enemyMissingPings': player.get('enemyMissingPings'),
            'enemyVisionPings': player.get('enemyVisionPings'),
            'holdPings': player.get('holdPings'),
            'getBackPings': player.get('getBackPings'),
            'needVisionPings': player.get('needVisionPings'),
            'onMyWayPings': player.get('onMyWayPings'),
            'pushPings': player.get('pushPings'),
            'visionClearedPings': player.get('visionClearedPings'),
        }

        # Flatten Perks (Runes)
        perks = player.get('perks', {})
        if perks:
            # Stat Shards (statPerks)
            stat_perks = perks.get('statPerks', {})
            if stat_perks:
                row['perk_stat_defense'] = stat_perks.get('defense')
                row['perk_stat_flex'] = stat_perks.get('flex')
                row['perk_stat_offense'] = stat_perks.get('offense')
            
            # Rune Styles (Primary & Secondary Trees + Selections)
            styles = perks.get('styles', [])
            for style_idx, style in enumerate(styles):
                if style_idx == 0:
                    prefix = 'perk_primary'
                elif style_idx == 1:
                    prefix = 'perk_secondary'
                else:
                    prefix = f'perk_extra_{style_idx}'
                
                row[f'{prefix}_style'] = style.get('style')
                row[f'{prefix}_style_description'] = style.get('description', '')
                
                selections = style.get('selections', [])
                for sel_idx, selection in enumerate(selections):
                    rune_num = sel_idx + 1
                    row[f'{prefix}_rune_{rune_num}'] = selection.get('perk')
                    row[f'{prefix}_rune_{rune_num}_var1'] = selection.get('var1')
                    row[f'{prefix}_rune_{rune_num}_var2'] = selection.get('var2')
                    row[f'{prefix}_rune_{rune_num}_var3'] = selection.get('var3')
        
        # Flatten Challenges
        challenges = player.get('challenges', {})
        if challenges:
            for challenge_key, challenge_value in challenges.items():
                if challenge_key in EXCLUDE_CHALLENGES:
                    continue
                row[f'challenge_{challenge_key}'] = challenge_value

        player_rows.append(row)
    
    return player_rows


def collect_and_flatten_match_data(region='europe', sub_region='eun1', matches_per_division=None):
    """Read merged match IDs, fetch data, flatten, and save to CSV files.
    
    Args:
        region: API region routing
        sub_region: Platform sub-region
        matches_per_division: dict or int. Number of matches to process per division.
    """
    
    if matches_per_division is None:
        matches_per_division = {
            'IRON': 50, 'BRONZE': 50, 'SILVER': 50, 'GOLD': 50,
            'PLATINUM': 50, 'EMERALD': 50, 'DIAMOND': 50,
            'MASTER': 200, 'GRANDMASTER': 200, 'CHALLENGER': 200
        }
    elif isinstance(matches_per_division, int):
        matches_per_division = {tier: matches_per_division for tier in 
                               ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND', 'MASTER', 'GRANDMASTER', 'CHALLENGER']}
    
    limiter = RateLimiter()
    
    # Load merged match IDs
    current_dir = os.getcwd()
    data_path = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'match ids'))
    
    all_files = [f for f in os.listdir(data_path) if f.startswith('Merged_Match_IDs_') and f.endswith('.csv')]
    if not all_files:
        print("No merged match IDs file found")
        return None, None
    
    latest_file = sorted(all_files, reverse=True)[0]
    df = pd.read_csv(os.path.join(data_path, latest_file))
    print(f"Loaded {len(df)} match IDs from {latest_file}")
    
    # Get unique divisions
    divisions = df['division'].unique()
    
    # Initialize lists to store all data
    all_team_rows = []
    all_player_rows = []
    total_processed = 0
    
    for division in divisions:
        tier = division.split('_')[0]
        target = matches_per_division.get(tier, 50)
        
        div_df = df[df['division'] == division].head(target)
        
        if div_df.empty:
            print(f"\n{division}: No match IDs found")
            continue
        
        print(f"\n{division} (fetching {len(div_df)} matches):")
        
        for idx, row in div_df.iterrows():
            match_id = row['match_id']
            total_processed += 1
            
            try:
                limiter.wait_if_needed()
                match_data = get_match_data_from_match_id(region=region, sub_region=sub_region, matchId=match_id)
                limiter.add_request()
                
                # Dont add this part cause their is lots of matches that contains 2 rank (e.g., gold and plat)
                # Add tier/rank/division info from the match IDs file
                tier_info = {
                    'tier': row['tier'],
                    'rank': row['rank'],
                    'division': row['division']
                }
                
                # Flatten team data
                team_rows = flatten_team_data(match_data, match_id)
                for team_row in team_rows:
                    team_row.update(tier_info)
                all_team_rows.extend(team_rows)
                
                # Flatten player data
                player_rows = flatten_player_data(match_data, match_id)
                for player_row in player_rows:
                    player_row.update(tier_info)
                all_player_rows.extend(player_rows)
                
                if total_processed % 10 == 0:
                    print(f"  Processed {total_processed} matches total...")
                
            except Exception as e:
                print(f"  Error processing {match_id}: {e}")
                time.sleep(5)
                continue
        
        print(f"  Completed {division}")
    
    # Create DataFrames
    teams_stats_merged = pd.DataFrame(all_team_rows)
    match_data_merged = pd.DataFrame(all_player_rows)
    
    # Create output directory
    output_dir = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'flattened'))
    os.makedirs(output_dir, exist_ok=True)
    
    # Save to CSV
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    teams_file = os.path.join(output_dir, f'teams_stats_merged_{timestamp}.csv')
    players_file = os.path.join(output_dir, f'match_data_merged_{timestamp}.csv')
    
    teams_stats_merged.to_csv(teams_file, index=False)
    match_data_merged.to_csv(players_file, index=False)
    
    print(f"\n{'='*60}")
    print(f"Processing complete!")
    print(f"Total matches processed: {total_processed}")
    print(f"Teams stats: {len(teams_stats_merged)} rows -> {teams_file}")
    print(f"Match data: {len(match_data_merged)} rows -> {players_file}")
    print(f"{'='*60}")
    
    return teams_stats_merged, match_data_merged


# Run the collection and flattening
if __name__ == "__main__":
    # Make sure to set your API key before running
    # api_key = 'RGAPI-YOUR-KEY-HERE'
    
    teams_stats, match_data = collect_and_flatten_match_data(
        region='europe',
        sub_region='eun1'
    )

Loaded 2000 match IDs from Merged_Match_IDs_20260613_160406.csv

IRON_I (fetching 50 matches):
  Processed 10 matches total...
  Processed 20 matches total...
  Processed 30 matches total...
  Processed 40 matches total...
  Processed 50 matches total...
  Completed IRON_I

IRON_II (fetching 50 matches):
  Processed 60 matches total...
  Processed 70 matches total...
  Processed 80 matches total...
  Processed 90 matches total...
  Processed 100 matches total...
  Completed IRON_II

IRON_III (fetching 50 matches):
  Processed 110 matches total...
  Processed 120 matches total...
  Processed 130 matches total...
  Processed 140 matches total...
  Processed 150 matches total...
  Completed IRON_III

IRON_IV (fetching 50 matches):
  Processed 160 matches total...
  Processed 170 matches total...
Long-term rate limit reached. Waiting 1.09s...
Long-term rate limit reached. Waiting 4.93s...
Long-term rate limit reached. Waiting 2.39s...
Long-term rate limit reached. Waiting 2.63s...
Long-ter

# HAS RETRY LOGIC HERE (FOR FAILED MATCHES)
# Reads match ids and extracts match data and saves it to a csv locally

In [ ]:
import pandas as pd
import requests
import time
import os
from datetime import datetime

# Your API key (make sure to set this)
api_key = 'YOUR_API_KEY_HERE'

class RateLimiter:
    def __init__(self):
        self.short_term_requests = []
        self.long_term_requests = []
        self.short_window = 1
        self.long_window = 120
        self.short_limit = 20
        self.long_limit = 100
    
    def wait_if_needed(self):
        current_time = time.time()
        self.short_term_requests = [t for t in self.short_term_requests if current_time - t < self.short_window]
        self.long_term_requests = [t for t in self.long_term_requests if current_time - t < self.long_window]
        
        if len(self.short_term_requests) >= self.short_limit:
            sleep_time = self.short_term_requests[0] + self.short_window - current_time
            if sleep_time > 0:
                print(f"Short-term rate limit reached. Waiting {sleep_time:.2f}s...")
                time.sleep(sleep_time)
                return self.wait_if_needed()
        
        if len(self.long_term_requests) >= self.long_limit:
            sleep_time = self.long_term_requests[0] + self.long_window - current_time
            if sleep_time > 0:
                print(f"Long-term rate limit reached. Waiting {sleep_time:.2f}s...")
                time.sleep(sleep_time)
                return self.wait_if_needed()
    
    def add_request(self):
        current_time = time.time()
        self.short_term_requests.append(current_time)
        self.long_term_requests.append(current_time)


def get_match_data_with_retry(region, sub_region, match_id, max_retries=3, limiter=None):
    """Fetch match data with retry logic for failed requests.
    
    Args:
        region: API region routing
        sub_region: Platform sub-region  
        match_id: Match ID to fetch
        max_retries: Maximum number of retry attempts
        limiter: RateLimiter instance
    
    Returns:
        dict: Match data if successful, None if all retries failed
    """
    root_url = f'https://{region}.api.riotgames.com/'
    endpoint = f'lol/match/v5/matches/{match_id}'
    
    for attempt in range(max_retries):
        try:
            if limiter:
                limiter.wait_if_needed()
            
            response = requests.get(root_url + endpoint + '?api_key=' + api_key, timeout=30)
            
            # Check for rate limiting (429)
            if response.status_code == 429:
                retry_after = int(response.headers.get('Retry-After', 10))
                print(f"  Rate limited (429). Waiting {retry_after}s...")
                time.sleep(retry_after)
                continue
            
            # Check for other bad status codes
            if response.status_code == 404:
                print(f"  Match {match_id} not found (404). Skipping.")
                return None
            
            if response.status_code != 200:
                print(f"  HTTP {response.status_code} for {match_id}. Retrying...")
                time.sleep(2 ** attempt)  # Exponential backoff
                continue
            
            if limiter:
                limiter.add_request()
            
            return response.json()
            
        except (requests.exceptions.ConnectionError, 
                requests.exceptions.Timeout,
                ConnectionResetError) as e:
            if attempt < max_retries - 1:
                wait_time = (2 ** attempt) * 5  # Exponential backoff: 5, 10, 20 seconds
                print(f"  Connection error for {match_id} (attempt {attempt + 1}/{max_retries}): {type(e).__name__}")
                print(f"    Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
            else:
                print(f"  Failed all {max_retries} attempts for {match_id}: {type(e).__name__}")
                return None
                
        except Exception as e:
            print(f"  Unexpected error for {match_id}: {type(e).__name__}: {e}")
            return None
    
    return None


def flatten_team_data(match_data, match_id):
    """Flatten team-level data from match JSON."""
    teams = match_data['info']['teams']
    team_rows = []

    for team in teams:
        teamid = team['teamId']
        win = team['win']

        # Flatten Bans
        bans = team['bans']
        ban_dict = {}
        for i, ban in enumerate(bans):
            ban_dict[f'ban_{i+1}_championId'] = ban['championId']
        
        # Flatten Objectives
        obj = team['objectives']
        obj_dict = {}
        for objective_name, objective_data in obj.items():
            obj_dict[f'obj_{objective_name}_first'] = objective_data['first']
            obj_dict[f'obj_{objective_name}_kills'] = objective_data['kills']
        
        # Combine Everything
        team_row = {
            'matchId': match_id,
            'gameDuration': match_data['info']['gameDuration'],
            'platformId': match_data['info']['platformId'],
            'queueId': match_data['info']['queueId'],
            'teamId': teamid,
            'win': win
        }
        team_row.update(ban_dict)
        team_row.update(obj_dict)
        
        team_rows.append(team_row)
    
    return team_rows


def flatten_player_data(match_data, match_id):
    """Flatten player-level data from match JSON."""
    
    # ChallengeDto Related Variables - Fields to EXCLUDE (non-Summoner's Rift modes)
    EXCLUDE_CHALLENGES = {
        'killsOnRecentlyHealedByAramPack', 'poroExplosions', 'snowballsHit',
        'abilityUses', 'fistBumpParticipation',
        'SWARM_DefeatAatrox', 'SWARM_DefeatBriar', 'SWARM_DefeatMiniBosses',
        'SWARM_EvolveWeapon', 'SWARM_Have3Passives', 'SWARM_KillEnemy',
        'SWARM_PickupGold', 'SWARM_ReachLevel50', 'SWARM_Survive15Min',
        'SWARM_WinWith5EvolvedWeapons', 'dancedWithRiftHerald',
        'legendaryItemUsed', 'InfernalScalePickup',
    }

    
    
    players = match_data['info']['participants']
    player_rows = []

    for player in players:
        row = {
            # Core Identity Fields
            'matchId': match_id,
            'gameDuration': match_data['info']['gameDuration'],
            'platformId': match_data['info']['platformId'],
            'queueId': match_data['info']['queueId'],
            'teamId': player.get('teamId'),
            'participantId': player.get('participantId'),
            'puuid': player.get('puuid'),
            'riotIdGameName': player.get('riotIdGameName'),
            'riotIdTagline': player.get('riotIdTagline'),
            'profileIcon': player.get('profileIcon'),
            'summonerLevel': player.get('summonerLevel'),
            
            # Team & Position
            'teamPosition': player.get('teamPosition'),
            
            # Champion Info
            'championId': player.get('championId'),
            'championName': player.get('championName'),
            'champLevel': player.get('champLevel'),
            
            # KDA
            'kills': player.get('kills'),
            'deaths': player.get('deaths'),
            'assists': player.get('assists'),
            
            # Multi-Kills
            'doubleKills': player.get('doubleKills'),
            'tripleKills': player.get('tripleKills'),
            'quadraKills': player.get('quadraKills'),
            'pentaKills': player.get('pentaKills'),
            'largestMultiKill': player.get('largestMultiKill'),
            'largestKillingSpree': player.get('largestKillingSpree'),
            'killingSprees': player.get('killingSprees'),
            
            # First Blood / Tower
            'firstBloodKill': player.get('firstBloodKill'),
            'firstBloodAssist': player.get('firstBloodAssist'),
            'firstTowerKill': player.get('firstTowerKill'),
            'firstTowerAssist': player.get('firstTowerAssist'),
            
            # Damage Dealt
            'totalDamageDealt': player.get('totalDamageDealt'),
            'totalDamageDealtToChampions': player.get('totalDamageDealtToChampions'),
            'physicalDamageDealt': player.get('physicalDamageDealt'),
            'physicalDamageDealtToChampions': player.get('physicalDamageDealtToChampions'),
            'magicDamageDealt': player.get('magicDamageDealt'),
            'magicDamageDealtToChampions': player.get('magicDamageDealtToChampions'),
            'trueDamageDealt': player.get('trueDamageDealt'),
            'trueDamageDealtToChampions': player.get('trueDamageDealtToChampions'),
            'damageDealtToBuildings': player.get('damageDealtToBuildings'),
            'damageDealtToObjectives': player.get('damageDealtToObjectives'),
            'damageDealtToTurrets': player.get('damageDealtToTurrets'),
            'largestCriticalStrike': player.get('largestCriticalStrike'),
            
            # Damage Taken & Mitigated
            'totalDamageTaken': player.get('totalDamageTaken'),
            'physicalDamageTaken': player.get('physicalDamageTaken'),
            'magicDamageTaken': player.get('magicDamageTaken'),
            'trueDamageTaken': player.get('trueDamageTaken'),
            'damageSelfMitigated': player.get('damageSelfMitigated'),
            
            # Healing & Shielding
            'totalHeal': player.get('totalHeal'),
            'totalHealsOnTeammates': player.get('totalHealsOnTeammates'),
            'totalUnitsHealed': player.get('totalUnitsHealed'),
            'totalDamageShieldedOnTeammates': player.get('totalDamageShieldedOnTeammates'),
            
            # Gold & Economy
            'goldEarned': player.get('goldEarned'),
            'goldSpent': player.get('goldSpent'),
            'consumablesPurchased': player.get('consumablesPurchased'),
            'itemsPurchased': player.get('itemsPurchased'),
            'bountyLevel': player.get('bountyLevel'),
            
            # Items (0-6 slots)
            'item0': player.get('item0'),
            'item1': player.get('item1'),
            'item2': player.get('item2'),
            'item3': player.get('item3'),
            'item4': player.get('item4'),
            'item5': player.get('item5'),
            'item6': player.get('item6'),
            
            # Summoner Spells
            'summoner1Id': player.get('summoner1Id'),
            'summoner1Casts': player.get('summoner1Casts'),
            'summoner2Id': player.get('summoner2Id'),
            'summoner2Casts': player.get('summoner2Casts'),
            
            # Ability Casts
            'spell1Casts': player.get('spell1Casts'),
            'spell2Casts': player.get('spell2Casts'),
            'spell3Casts': player.get('spell3Casts'),
            'spell4Casts': player.get('spell4Casts'),
            
            # Minions & Jungle
            'totalMinionsKilled': player.get('totalMinionsKilled'),
            'neutralMinionsKilled': player.get('neutralMinionsKilled'),
            'totalAllyJungleMinionsKilled': player.get('totalAllyJungleMinionsKilled'),
            'totalEnemyJungleMinionsKilled': player.get('totalEnemyJungleMinionsKilled'),
            
            # Objectives
            'baronKills': player.get('baronKills'),
            'dragonKills': player.get('dragonKills'),
            'inhibitorKills': player.get('inhibitorKills'),
            'inhibitorTakedowns': player.get('inhibitorTakedowns'),
            'inhibitorsLost': player.get('inhibitorsLost'),
            'turretKills': player.get('turretKills'),
            'turretTakedowns': player.get('turretTakedowns'),
            'turretsLost': player.get('turretsLost'),
            'nexusKills': player.get('nexusKills'),
            'nexusTakedowns': player.get('nexusTakedowns'),
            'nexusLost': player.get('nexusLost'),
            'objectivesStolen': player.get('objectivesStolen'),
            'objectivesStolenAssists': player.get('objectivesStolenAssists'),
            
            # Vision
            'visionScore': player.get('visionScore'),
            'wardsPlaced': player.get('wardsPlaced'),
            'wardsKilled': player.get('wardsKilled'),
            'sightWardsBoughtInGame': player.get('sightWardsBoughtInGame'),
            'visionWardsBoughtInGame': player.get('visionWardsBoughtInGame'),
            'detectorWardsPlaced': player.get('detectorWardsPlaced'),
            
            # CC (Crowd Control)
            'timeCCingOthers': player.get('timeCCingOthers'),
            'totalTimeCCDealt': player.get('totalTimeCCDealt'),
            
            # Time
            'timePlayed': player.get('timePlayed'),
            'totalTimeSpentDead': player.get('totalTimeSpentDead'),
            'longestTimeSpentLiving': player.get('longestTimeSpentLiving'),
            
            # Pings (Communication)
            'allInPings': player.get('allInPings'),
            'assistMePings': player.get('assistMePings'),
            'commandPings': player.get('commandPings'),
            'enemyMissingPings': player.get('enemyMissingPings'),
            'enemyVisionPings': player.get('enemyVisionPings'),
            'holdPings': player.get('holdPings'),
            'getBackPings': player.get('getBackPings'),
            'needVisionPings': player.get('needVisionPings'),
            'onMyWayPings': player.get('onMyWayPings'),
            'pushPings': player.get('pushPings'),
            'visionClearedPings': player.get('visionClearedPings'),
        }

        # Flatten Perks (Runes)
        perks = player.get('perks', {})
        if perks:
            # Stat Shards (statPerks)
            stat_perks = perks.get('statPerks', {})
            if stat_perks:
                row['perk_stat_defense'] = stat_perks.get('defense')
                row['perk_stat_flex'] = stat_perks.get('flex')
                row['perk_stat_offense'] = stat_perks.get('offense')
            
            # Rune Styles (Primary & Secondary Trees + Selections)
            styles = perks.get('styles', [])
            for style_idx, style in enumerate(styles):
                if style_idx == 0:
                    prefix = 'perk_primary'
                elif style_idx == 1:
                    prefix = 'perk_secondary'
                else:
                    prefix = f'perk_extra_{style_idx}'
                
                row[f'{prefix}_style'] = style.get('style')
                row[f'{prefix}_style_description'] = style.get('description', '')
                
                selections = style.get('selections', [])
                for sel_idx, selection in enumerate(selections):
                    rune_num = sel_idx + 1
                    row[f'{prefix}_rune_{rune_num}'] = selection.get('perk')
                    row[f'{prefix}_rune_{rune_num}_var1'] = selection.get('var1')
                    row[f'{prefix}_rune_{rune_num}_var2'] = selection.get('var2')
                    row[f'{prefix}_rune_{rune_num}_var3'] = selection.get('var3')
        
        # Flatten Challenges
        challenges = player.get('challenges', {})
        if challenges:
            for challenge_key, challenge_value in challenges.items():
                if challenge_key in EXCLUDE_CHALLENGES:
                    continue
                row[f'challenge_{challenge_key}'] = challenge_value

        player_rows.append(row)
    
    return player_rows


def collect_and_flatten_match_data(region='europe', sub_region='eun1', matches_per_division=None, max_retries=3):
    """Read merged match IDs, fetch data, flatten, and save to CSV files.
    
    Args:
        region: API region routing
        sub_region: Platform sub-region
        matches_per_division: dict or int. Number of matches to process per division.
        max_retries: Maximum number of retry attempts for failed requests
    """
    
    if matches_per_division is None:
        matches_per_division = {
            'IRON': 50, 'BRONZE': 50, 'SILVER': 50, 'GOLD': 50,
            'PLATINUM': 50, 'EMERALD': 50, 'DIAMOND': 50,
            'MASTER': 200, 'GRANDMASTER': 200, 'CHALLENGER': 200
        }
    elif isinstance(matches_per_division, int):
        matches_per_division = {tier: matches_per_division for tier in 
                               ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND', 'MASTER', 'GRANDMASTER', 'CHALLENGER']}
    
    limiter = RateLimiter()
    
    # Load merged match IDs
    current_dir = os.getcwd()
    data_path = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'match ids'))
    
    all_files = [f for f in os.listdir(data_path) if f.startswith('Merged_Match_IDs_') and f.endswith('.csv')]
    if not all_files:
        print("No merged match IDs file found")
        return None, None
    
    latest_file = sorted(all_files, reverse=True)[0]
    df = pd.read_csv(os.path.join(data_path, latest_file))
    print(f"Loaded {len(df)} match IDs from {latest_file}")
    
    # Get unique divisions
    divisions = df['division'].unique()
    
    # Initialize lists to store all data and track failures
    all_team_rows = []
    all_player_rows = []
    failed_matches = []
    total_processed = 0
    successful_matches = 0
    
    for division in divisions:
        tier = division.split('_')[0]
        target = matches_per_division.get(tier, 50)
        
        div_df = df[df['division'] == division].head(target)
        
        if div_df.empty:
            print(f"\n{division}: No match IDs found")
            continue
        
        print(f"\n{division} (fetching {len(div_df)} matches):")
        
        for idx, row in div_df.iterrows():
            match_id = row['match_id']
            total_processed += 1
            
            # Fetch with retry logic
            match_data = get_match_data_with_retry(
                region=region, 
                sub_region=sub_region, 
                match_id=match_id, 
                max_retries=max_retries,
                limiter=limiter
            )
            
            if match_data is None:
                failed_matches.append({
                    'match_id': match_id,
                    'tier': row['tier'],
                    'rank': row['rank'],
                    'division': row['division']
                })
                continue
            
            successful_matches += 1
            
            try:
                # Add tier/rank/division info from the match IDs file
                tier_info = {
                    'tier': row['tier'],
                    'rank': row['rank'],
                    'division': row['division']
                }
                
                # Flatten team data
                team_rows = flatten_team_data(match_data, match_id)
                for team_row in team_rows:
                    team_row.update(tier_info)
                all_team_rows.extend(team_rows)
                
                # Flatten player data
                player_rows = flatten_player_data(match_data, match_id)
                for player_row in player_rows:
                    player_row.update(tier_info)
                all_player_rows.extend(player_rows)
                
                if total_processed % 10 == 0:
                    print(f"  Processed {total_processed} matches ({successful_matches} successful, {len(failed_matches)} failed)...")
                
            except Exception as e:
                print(f"  Error flattening {match_id}: {e}")
                failed_matches.append({
                    'match_id': match_id,
                    'tier': row['tier'],
                    'rank': row['rank'],
                    'division': row['division']
                })
                continue
        
        print(f"  Completed {division}")
    
    # Create DataFrames
    teams_stats_merged = pd.DataFrame(all_team_rows)
    match_data_merged = pd.DataFrame(all_player_rows)
    
    # Create output directory
    output_dir = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'processed'))
    os.makedirs(output_dir, exist_ok=True)
    
    # Save to CSV
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    teams_file = os.path.join(output_dir, f'teams_stats_merged_{timestamp}.csv')
    players_file = os.path.join(output_dir, f'match_data_merged_{timestamp}.csv')
    
    teams_stats_merged.to_csv(teams_file, index=False)
    match_data_merged.to_csv(players_file, index=False)
    
    # Save failed matches if any
    if failed_matches:
        failed_df = pd.DataFrame(failed_matches)
        failed_file = os.path.join(output_dir, f'failed_matches_{timestamp}.csv')
        failed_df.to_csv(failed_file, index=False)
        print(f"\n  Failed matches saved to: {failed_file}")
    
    print(f"\n{'='*60}")
    print(f"Processing complete!")
    print(f"Total matches attempted: {total_processed}")
    print(f"Successful: {successful_matches}")
    print(f"Failed: {len(failed_matches)}")
    print(f"Success rate: {successful_matches/total_processed*100:.1f}%")
    print(f"Teams stats: {len(teams_stats_merged)} rows -> {teams_file}")
    print(f"Match data: {len(match_data_merged)} rows -> {players_file}")
    print(f"{'='*60}")
    
    # Print failed match IDs for reference
    if failed_matches:
        print(f"\nFailed match IDs ({len(failed_matches)}):")
        for fm in failed_matches[:10]:  # Show first 10
            print(f"  - {fm['match_id']} ({fm['division']})")
        if len(failed_matches) > 10:
            print(f"  ... and {len(failed_matches) - 10} more")
    
    return teams_stats_merged, match_data_merged, failed_matches


# Optional: Function to retry failed matches only
def retry_failed_matches(failed_csv_path, region='europe', sub_region='eun1', max_retries=5):
    """Retry fetching matches that previously failed.
    
    Args:
        failed_csv_path: Path to the CSV file with failed matches
        region: API region routing
        sub_region: Platform sub-region
        max_retries: Maximum number of retry attempts
    """
    print(f"Loading failed matches from: {failed_csv_path}")
    failed_df = pd.read_csv(failed_csv_path)
    print(f"Found {len(failed_df)} failed matches to retry")
    
    limiter = RateLimiter()
    all_team_rows = []
    all_player_rows = []
    still_failed = []
    successful = 0
    
    for idx, row in failed_df.iterrows():
        match_id = row['match_id']
        
        print(f"\nRetrying {match_id} ({row['division']}) - Attempt {idx+1}/{len(failed_df)}")
        
        match_data = get_match_data_with_retry(
            region=region,
            sub_region=sub_region,
            match_id=match_id,
            max_retries=max_retries,
            limiter=limiter
        )
        
        if match_data is None:
            still_failed.append(row.to_dict())
            continue
        
        successful += 1
        
        try:
            team_rows = flatten_team_data(match_data, match_id)
            all_team_rows.extend(team_rows)
            
            player_rows = flatten_player_data(match_data, match_id)
            all_player_rows.extend(player_rows)
            
        except Exception as e:
            print(f"  Error flattening {match_id}: {e}")
            still_failed.append(row.to_dict())
            continue
    
    # Save newly retrieved data
    if all_team_rows:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_dir = os.path.dirname(failed_csv_path)
        
        teams_df = pd.DataFrame(all_team_rows)
        players_df = pd.DataFrame(all_player_rows)
        
        teams_file = os.path.join(output_dir, f'teams_stats_retry_{timestamp}.csv')
        players_file = os.path.join(output_dir, f'match_data_retry_{timestamp}.csv')
        
        teams_df.to_csv(teams_file, index=False)
        players_df.to_csv(players_file, index=False)
        
        print(f"\nSaved retry data:")
        print(f"  Teams: {teams_file}")
        print(f"  Players: {players_file}")
    
    # Save still-failed matches
    if still_failed:
        still_failed_df = pd.DataFrame(still_failed)
        still_failed_file = os.path.join(output_dir, f'still_failed_matches_{timestamp}.csv')
        still_failed_df.to_csv(still_failed_file, index=False)
        print(f"Still failed ({len(still_failed)}): {still_failed_file}")
    
    print(f"\nRetry complete: {successful} recovered, {len(still_failed)} still failed")
    
    return pd.DataFrame(all_team_rows) if all_team_rows else None, \
           pd.DataFrame(all_player_rows) if all_player_rows else None, \
           still_failed


# Run the collection and flattening
if __name__ == "__main__":
    # Make sure to set your API key before running
    # api_key = 'RGAPI-YOUR-KEY-HERE'
    
    teams_stats, match_data, failed_matches = collect_and_flatten_match_data(
        region='europe',
        sub_region='eun1',
        max_retries=3
    )

# NO TIER RANK AND DIVISION DATA ADDED TO THE CSV FILES
## Also failed match lookups are stored into a CSV

In [6]:
import pandas as pd
import requests
import time
import os
from datetime import datetime


class RateLimiter:
    def __init__(self):
        self.short_term_requests = []
        self.long_term_requests = []
        self.short_window = 1
        self.long_window = 120
        self.short_limit = 20
        self.long_limit = 100
    
    def wait_if_needed(self):
        current_time = time.time()
        self.short_term_requests = [t for t in self.short_term_requests if current_time - t < self.short_window]
        self.long_term_requests = [t for t in self.long_term_requests if current_time - t < self.long_window]
        
        if len(self.short_term_requests) >= self.short_limit:
            sleep_time = self.short_term_requests[0] + self.short_window - current_time
            if sleep_time > 0:
                print(f"Short-term rate limit reached. Waiting {sleep_time:.2f}s...")
                time.sleep(sleep_time)
                return self.wait_if_needed()
        
        if len(self.long_term_requests) >= self.long_limit:
            sleep_time = self.long_term_requests[0] + self.long_window - current_time
            if sleep_time > 0:
                print(f"Long-term rate limit reached. Waiting {sleep_time:.2f}s...")
                time.sleep(sleep_time)
                return self.wait_if_needed()
    
    def add_request(self):
        current_time = time.time()
        self.short_term_requests.append(current_time)
        self.long_term_requests.append(current_time)


def get_match_data_with_retry(region, sub_region, match_id, max_retries=3, limiter=None):
    """Fetch match data with retry logic for failed requests.
    
    Args:
        region: API region routing
        sub_region: Platform sub-region  
        match_id: Match ID to fetch
        max_retries: Maximum number of retry attempts
        limiter: RateLimiter instance
    
    Returns:
        dict: Match data if successful, None if all retries failed
    """
    root_url = f'https://{region}.api.riotgames.com/'
    endpoint = f'lol/match/v5/matches/{match_id}'
    
    for attempt in range(max_retries):
        try:
            if limiter:
                limiter.wait_if_needed()
            
            response = requests.get(root_url + endpoint + '?api_key=' + api_key, timeout=30)
            
            # Check for rate limiting (429)
            if response.status_code == 429:
                retry_after = int(response.headers.get('Retry-After', 10))
                print(f"  Rate limited (429). Waiting {retry_after}s...")
                time.sleep(retry_after)
                continue
            
            # Check for other bad status codes
            if response.status_code == 404:
                print(f"  Match {match_id} not found (404). Skipping.")
                return None
            
            if response.status_code != 200:
                print(f"  HTTP {response.status_code} for {match_id}. Retrying...")
                time.sleep(2 ** attempt)  # Exponential backoff
                continue
            
            if limiter:
                limiter.add_request()
            
            return response.json()
            
        except (requests.exceptions.ConnectionError, 
                requests.exceptions.Timeout,
                ConnectionResetError) as e:
            if attempt < max_retries - 1:
                wait_time = (2 ** attempt) * 5  # Exponential backoff: 5, 10, 20 seconds
                print(f"  Connection error for {match_id} (attempt {attempt + 1}/{max_retries}): {type(e).__name__}")
                print(f"    Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
            else:
                print(f"  Failed all {max_retries} attempts for {match_id}: {type(e).__name__}")
                return None
                
        except Exception as e:
            print(f"  Unexpected error for {match_id}: {type(e).__name__}: {e}")
            return None
    
    return None


def flatten_team_data(match_data, match_id):
    """Flatten team-level data from match JSON."""
    teams = match_data['info']['teams']
    team_rows = []

    for team in teams:
        teamid = team['teamId']
        win = team['win']

        # Flatten Bans
        bans = team['bans']
        ban_dict = {}
        for i, ban in enumerate(bans):
            ban_dict[f'ban_{i+1}_championId'] = ban['championId']
        
        # Flatten Objectives
        obj = team['objectives']
        obj_dict = {}
        for objective_name, objective_data in obj.items():
            obj_dict[f'obj_{objective_name}_first'] = objective_data['first']
            obj_dict[f'obj_{objective_name}_kills'] = objective_data['kills']
        
        # Combine Everything
        team_row = {
            'matchId': match_id,
            'gameDuration': match_data['info']['gameDuration'],
            'platformId': match_data['info']['platformId'],
            'queueId': match_data['info']['queueId'],
            'teamId': teamid,
            'win': win
        }
        team_row.update(ban_dict)
        team_row.update(obj_dict)
        
        team_rows.append(team_row)
    
    return team_rows


def flatten_player_data(match_data, match_id):
    """Flatten player-level data from match JSON."""
    
    # ChallengeDto Related Variables - Fields to EXCLUDE (non-Summoner's Rift modes)
    EXCLUDE_CHALLENGES = {
        'killsOnRecentlyHealedByAramPack', 'poroExplosions', 'snowballsHit',
        'abilityUses', 'fistBumpParticipation',
        'SWARM_DefeatAatrox', 'SWARM_DefeatBriar', 'SWARM_DefeatMiniBosses',
        'SWARM_EvolveWeapon', 'SWARM_Have3Passives', 'SWARM_KillEnemy',
        'SWARM_PickupGold', 'SWARM_ReachLevel50', 'SWARM_Survive15Min',
        'SWARM_WinWith5EvolvedWeapons', 'dancedWithRiftHerald',
        'legendaryItemUsed', 'InfernalScalePickup',
    }
    
    players = match_data['info']['participants']
    player_rows = []

    for player in players:
        row = {
            # Core Identity Fields
            'matchId': match_id,
            'gameDuration': match_data['info']['gameDuration'],
            'platformId': match_data['info']['platformId'],
            'queueId': match_data['info']['queueId'],
            'teamId': player.get('teamId'),
            'participantId': player.get('participantId'),
            'puuid': player.get('puuid'),
            'riotIdGameName': player.get('riotIdGameName'),
            'riotIdTagline': player.get('riotIdTagline'),
            'profileIcon': player.get('profileIcon'),
            'summonerLevel': player.get('summonerLevel'),
            
            # Team & Position
            'teamPosition': player.get('teamPosition'),
            
            # Champion Info
            'championId': player.get('championId'),
            'championName': player.get('championName'),
            'champLevel': player.get('champLevel'),
            
            # KDA
            'kills': player.get('kills'),
            'deaths': player.get('deaths'),
            'assists': player.get('assists'),
            
            # Multi-Kills
            'doubleKills': player.get('doubleKills'),
            'tripleKills': player.get('tripleKills'),
            'quadraKills': player.get('quadraKills'),
            'pentaKills': player.get('pentaKills'),
            'largestMultiKill': player.get('largestMultiKill'),
            'largestKillingSpree': player.get('largestKillingSpree'),
            'killingSprees': player.get('killingSprees'),
            
            # First Blood / Tower
            'firstBloodKill': player.get('firstBloodKill'),
            'firstBloodAssist': player.get('firstBloodAssist'),
            'firstTowerKill': player.get('firstTowerKill'),
            'firstTowerAssist': player.get('firstTowerAssist'),
            
            # Damage Dealt
            'totalDamageDealt': player.get('totalDamageDealt'),
            'totalDamageDealtToChampions': player.get('totalDamageDealtToChampions'),
            'physicalDamageDealt': player.get('physicalDamageDealt'),
            'physicalDamageDealtToChampions': player.get('physicalDamageDealtToChampions'),
            'magicDamageDealt': player.get('magicDamageDealt'),
            'magicDamageDealtToChampions': player.get('magicDamageDealtToChampions'),
            'trueDamageDealt': player.get('trueDamageDealt'),
            'trueDamageDealtToChampions': player.get('trueDamageDealtToChampions'),
            'damageDealtToBuildings': player.get('damageDealtToBuildings'),
            'damageDealtToObjectives': player.get('damageDealtToObjectives'),
            'damageDealtToTurrets': player.get('damageDealtToTurrets'),
            'largestCriticalStrike': player.get('largestCriticalStrike'),
            
            # Damage Taken & Mitigated
            'totalDamageTaken': player.get('totalDamageTaken'),
            'physicalDamageTaken': player.get('physicalDamageTaken'),
            'magicDamageTaken': player.get('magicDamageTaken'),
            'trueDamageTaken': player.get('trueDamageTaken'),
            'damageSelfMitigated': player.get('damageSelfMitigated'),
            
            # Healing & Shielding
            'totalHeal': player.get('totalHeal'),
            'totalHealsOnTeammates': player.get('totalHealsOnTeammates'),
            'totalUnitsHealed': player.get('totalUnitsHealed'),
            'totalDamageShieldedOnTeammates': player.get('totalDamageShieldedOnTeammates'),
            
            # Gold & Economy
            'goldEarned': player.get('goldEarned'),
            'goldSpent': player.get('goldSpent'),
            'consumablesPurchased': player.get('consumablesPurchased'),
            'itemsPurchased': player.get('itemsPurchased'),
            'bountyLevel': player.get('bountyLevel'),
            
            # Items (0-6 slots)
            'item0': player.get('item0'),
            'item1': player.get('item1'),
            'item2': player.get('item2'),
            'item3': player.get('item3'),
            'item4': player.get('item4'),
            'item5': player.get('item5'),
            'item6': player.get('item6'),
            
            # Summoner Spells
            'summoner1Id': player.get('summoner1Id'),
            'summoner1Casts': player.get('summoner1Casts'),
            'summoner2Id': player.get('summoner2Id'),
            'summoner2Casts': player.get('summoner2Casts'),
            
            # Ability Casts
            'spell1Casts': player.get('spell1Casts'),
            'spell2Casts': player.get('spell2Casts'),
            'spell3Casts': player.get('spell3Casts'),
            'spell4Casts': player.get('spell4Casts'),
            
            # Minions & Jungle
            'totalMinionsKilled': player.get('totalMinionsKilled'),
            'neutralMinionsKilled': player.get('neutralMinionsKilled'),
            'totalAllyJungleMinionsKilled': player.get('totalAllyJungleMinionsKilled'),
            'totalEnemyJungleMinionsKilled': player.get('totalEnemyJungleMinionsKilled'),
            
            # Objectives
            'baronKills': player.get('baronKills'),
            'dragonKills': player.get('dragonKills'),
            'inhibitorKills': player.get('inhibitorKills'),
            'inhibitorTakedowns': player.get('inhibitorTakedowns'),
            'inhibitorsLost': player.get('inhibitorsLost'),
            'turretKills': player.get('turretKills'),
            'turretTakedowns': player.get('turretTakedowns'),
            'turretsLost': player.get('turretsLost'),
            'nexusKills': player.get('nexusKills'),
            'nexusTakedowns': player.get('nexusTakedowns'),
            'nexusLost': player.get('nexusLost'),
            'objectivesStolen': player.get('objectivesStolen'),
            'objectivesStolenAssists': player.get('objectivesStolenAssists'),
            
            # Vision
            'visionScore': player.get('visionScore'),
            'wardsPlaced': player.get('wardsPlaced'),
            'wardsKilled': player.get('wardsKilled'),
            'sightWardsBoughtInGame': player.get('sightWardsBoughtInGame'),
            'visionWardsBoughtInGame': player.get('visionWardsBoughtInGame'),
            'detectorWardsPlaced': player.get('detectorWardsPlaced'),
            
            # CC (Crowd Control)
            'timeCCingOthers': player.get('timeCCingOthers'),
            'totalTimeCCDealt': player.get('totalTimeCCDealt'),
            
            # Time
            'timePlayed': player.get('timePlayed'),
            'totalTimeSpentDead': player.get('totalTimeSpentDead'),
            'longestTimeSpentLiving': player.get('longestTimeSpentLiving'),
            
            # Pings (Communication)
            'allInPings': player.get('allInPings'),
            'assistMePings': player.get('assistMePings'),
            'commandPings': player.get('commandPings'),
            'enemyMissingPings': player.get('enemyMissingPings'),
            'enemyVisionPings': player.get('enemyVisionPings'),
            'holdPings': player.get('holdPings'),
            'getBackPings': player.get('getBackPings'),
            'needVisionPings': player.get('needVisionPings'),
            'onMyWayPings': player.get('onMyWayPings'),
            'pushPings': player.get('pushPings'),
            'visionClearedPings': player.get('visionClearedPings'),
        }

        # Flatten Perks (Runes)
        perks = player.get('perks', {})
        if perks:
            # Stat Shards (statPerks)
            stat_perks = perks.get('statPerks', {})
            if stat_perks:
                row['perk_stat_defense'] = stat_perks.get('defense')
                row['perk_stat_flex'] = stat_perks.get('flex')
                row['perk_stat_offense'] = stat_perks.get('offense')
            
            # Rune Styles (Primary & Secondary Trees + Selections)
            styles = perks.get('styles', [])
            for style_idx, style in enumerate(styles):
                if style_idx == 0:
                    prefix = 'perk_primary'
                elif style_idx == 1:
                    prefix = 'perk_secondary'
                else:
                    prefix = f'perk_extra_{style_idx}'
                
                row[f'{prefix}_style'] = style.get('style')
                row[f'{prefix}_style_description'] = style.get('description', '')
                
                selections = style.get('selections', [])
                for sel_idx, selection in enumerate(selections):
                    rune_num = sel_idx + 1
                    row[f'{prefix}_rune_{rune_num}'] = selection.get('perk')
                    row[f'{prefix}_rune_{rune_num}_var1'] = selection.get('var1')
                    row[f'{prefix}_rune_{rune_num}_var2'] = selection.get('var2')
                    row[f'{prefix}_rune_{rune_num}_var3'] = selection.get('var3')
        
        # Flatten Challenges
        challenges = player.get('challenges', {})
        if challenges:
            for challenge_key, challenge_value in challenges.items():
                if challenge_key in EXCLUDE_CHALLENGES:
                    continue
                row[f'challenge_{challenge_key}'] = challenge_value

        player_rows.append(row)
    
    return player_rows


def collect_and_flatten_match_data(region='europe', sub_region='eun1', matches_per_division=None, max_retries=3):
    """Read merged match IDs, fetch data, flatten, and save to CSV files.
    
    Args:
        region: API region routing
        sub_region: Platform sub-region
        matches_per_division: dict or int. Number of matches to process per division.
        max_retries: Maximum number of retry attempts for failed requests
    """
    
    if matches_per_division is None:
        matches_per_division = {
            'IRON': 50, 'BRONZE': 50, 'SILVER': 50, 'GOLD': 50,
            'PLATINUM': 50, 'EMERALD': 50, 'DIAMOND': 50,
            'MASTER': 200, 'GRANDMASTER': 200, 'CHALLENGER': 200
        }
    elif isinstance(matches_per_division, int):
        matches_per_division = {tier: matches_per_division for tier in 
                               ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND', 'MASTER', 'GRANDMASTER', 'CHALLENGER']}
    
    limiter = RateLimiter()
    
    # Load merged match IDs
    current_dir = os.getcwd()
    data_path = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'match ids'))
    
    all_files = [f for f in os.listdir(data_path) if f.startswith('Merged_Match_IDs_') and f.endswith('.csv')]
    if not all_files:
        print("No merged match IDs file found")
        return None, None, None
    
    latest_file = sorted(all_files, reverse=True)[0]
    df = pd.read_csv(os.path.join(data_path, latest_file))
    print(f"Loaded {len(df)} match IDs from {latest_file}")
    
    # Get unique divisions
    divisions = df['division'].unique()
    
    # Initialize lists to store all data and track failures
    all_team_rows = []
    all_player_rows = []
    failed_matches = []
    total_processed = 0
    successful_matches = 0
    
    for division in divisions:
        tier = division.split('_')[0]
        target = matches_per_division.get(tier, 50)
        
        div_df = df[df['division'] == division].head(target)
        
        if div_df.empty:
            print(f"\n{division}: No match IDs found")
            continue
        
        print(f"\n{division} (fetching {len(div_df)} matches):")
        
        for idx, row in div_df.iterrows():
            match_id = row['match_id']
            total_processed += 1
            
            # Fetch with retry logic
            match_data = get_match_data_with_retry(
                region=region, 
                sub_region=sub_region, 
                match_id=match_id, 
                max_retries=max_retries,
                limiter=limiter
            )
            
            if match_data is None:
                # Only save tier/rank/division to failed matches file, NOT to output CSVs
                failed_matches.append({
                    'match_id': match_id,
                    'tier': row['tier'],
                    'rank': row['rank'],
                    'division': row['division']
                })
                continue
            
            successful_matches += 1
            
            try:
                # Flatten team data (NO tier/rank/division added)
                team_rows = flatten_team_data(match_data, match_id)
                all_team_rows.extend(team_rows)
                
                # Flatten player data (NO tier/rank/division added)
                player_rows = flatten_player_data(match_data, match_id)
                all_player_rows.extend(player_rows)
                
                if total_processed % 10 == 0:
                    print(f"  Processed {total_processed} matches ({successful_matches} successful, {len(failed_matches)} failed)...")
                
            except Exception as e:
                print(f"  Error flattening {match_id}: {e}")
                failed_matches.append({
                    'match_id': match_id,
                    'tier': row['tier'],
                    'rank': row['rank'],
                    'division': row['division']
                })
                continue
        
        print(f"  Completed {division}")
    
    # Create DataFrames
    teams_stats_merged = pd.DataFrame(all_team_rows)
    match_data_merged = pd.DataFrame(all_player_rows)
    
    # Create output directory
    output_dir = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'flattened'))
    os.makedirs(output_dir, exist_ok=True)
    
    # Save to CSV
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    teams_file = os.path.join(output_dir, f'teams_stats_merged_{timestamp}.csv')
    players_file = os.path.join(output_dir, f'match_data_merged_{timestamp}.csv')
    
    teams_stats_merged.to_csv(teams_file, index=False)
    match_data_merged.to_csv(players_file, index=False)
    
    # Save failed matches in SEPARATE file with tier/rank/division info
    if failed_matches:
        failed_df = pd.DataFrame(failed_matches)
        failed_file = os.path.join(output_dir, f'failed_matches_{timestamp}.csv')
        failed_df.to_csv(failed_file, index=False)
        print(f"\n  Failed matches saved to: {failed_file}")
    
    print(f"\n{'='*60}")
    print(f"Processing complete!")
    print(f"Total matches attempted: {total_processed}")
    print(f"Successful: {successful_matches}")
    print(f"Failed: {len(failed_matches)}")
    print(f"Success rate: {successful_matches/total_processed*100:.1f}%")
    print(f"Teams stats: {len(teams_stats_merged)} rows -> {teams_file}")
    print(f"Match data: {len(match_data_merged)} rows -> {players_file}")
    print(f"{'='*60}")
    
    # Print failed match IDs for reference
    if failed_matches:
        print(f"\nFailed match IDs ({len(failed_matches)}):")
        for fm in failed_matches[:10]:  # Show first 10
            print(f"  - {fm['match_id']} ({fm['division']})")
        if len(failed_matches) > 10:
            print(f"  ... and {len(failed_matches) - 10} more")
    
    return teams_stats_merged, match_data_merged, failed_matches


# Optional: Function to retry failed matches only
def retry_failed_matches(failed_csv_path, region='europe', sub_region='eun1', max_retries=5):
    """Retry fetching matches that previously failed.
    
    Args:
        failed_csv_path: Path to the CSV file with failed matches
        region: API region routing
        sub_region: Platform sub-region
        max_retries: Maximum number of retry attempts
    """
    print(f"Loading failed matches from: {failed_csv_path}")
    failed_df = pd.read_csv(failed_csv_path)
    print(f"Found {len(failed_df)} failed matches to retry")
    
    limiter = RateLimiter()
    all_team_rows = []
    all_player_rows = []
    still_failed = []
    successful = 0
    
    for idx, row in failed_df.iterrows():
        match_id = row['match_id']
        
        print(f"\nRetrying {match_id} ({row['division']}) - Attempt {idx+1}/{len(failed_df)}")
        
        match_data = get_match_data_with_retry(
            region=region,
            sub_region=sub_region,
            match_id=match_id,
            max_retries=max_retries,
            limiter=limiter
        )
        
        if match_data is None:
            still_failed.append(row.to_dict())
            continue
        
        successful += 1
        
        try:
            # NO tier/rank/division added to output
            team_rows = flatten_team_data(match_data, match_id)
            all_team_rows.extend(team_rows)
            
            player_rows = flatten_player_data(match_data, match_id)
            all_player_rows.extend(player_rows)
            
        except Exception as e:
            print(f"  Error flattening {match_id}: {e}")
            still_failed.append(row.to_dict())
            continue
    
    # Save newly retrieved data
    if all_team_rows:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_dir = os.path.dirname(failed_csv_path)
        
        teams_df = pd.DataFrame(all_team_rows)
        players_df = pd.DataFrame(all_player_rows)
        
        teams_file = os.path.join(output_dir, f'teams_stats_retry_{timestamp}.csv')
        players_file = os.path.join(output_dir, f'match_data_retry_{timestamp}.csv')
        
        teams_df.to_csv(teams_file, index=False)
        players_df.to_csv(players_file, index=False)
        
        print(f"\nSaved retry data:")
        print(f"  Teams: {teams_file}")
        print(f"  Players: {players_file}")
    
    # Save still-failed matches
    if still_failed:
        still_failed_df = pd.DataFrame(still_failed)
        still_failed_file = os.path.join(output_dir, f'still_failed_matches_{timestamp}.csv')
        still_failed_df.to_csv(still_failed_file, index=False)
        print(f"Still failed ({len(still_failed)}): {still_failed_file}")
    
    print(f"\nRetry complete: {successful} recovered, {len(still_failed)} still failed")
    
    return pd.DataFrame(all_team_rows) if all_team_rows else None, \
           pd.DataFrame(all_player_rows) if all_player_rows else None, \
           still_failed


# Run the collection and flattening
if __name__ == "__main__":
    # Make sure to set your API key before running
    # api_key = 'RGAPI-YOUR-KEY-HERE'
    
    teams_stats, match_data, failed_matches = collect_and_flatten_match_data(
        region='europe',
        sub_region='eun1',
        max_retries=3
    )

Loaded 2000 match IDs from Merged_Match_IDs_20260613_160406.csv

IRON_I (fetching 50 matches):
  Processed 10 matches (10 successful, 0 failed)...
  Processed 20 matches (20 successful, 0 failed)...
  Processed 30 matches (30 successful, 0 failed)...
  Processed 40 matches (40 successful, 0 failed)...
  Processed 50 matches (50 successful, 0 failed)...
  Completed IRON_I

IRON_II (fetching 50 matches):
  Processed 60 matches (60 successful, 0 failed)...
  Processed 70 matches (70 successful, 0 failed)...
  Processed 80 matches (80 successful, 0 failed)...
  Processed 90 matches (90 successful, 0 failed)...
  Processed 100 matches (100 successful, 0 failed)...
  Completed IRON_II

IRON_III (fetching 50 matches):
Long-term rate limit reached. Waiting 29.93s...
  Processed 110 matches (110 successful, 0 failed)...
  Processed 120 matches (120 successful, 0 failed)...
  Processed 130 matches (130 successful, 0 failed)...
  Processed 140 matches (140 successful, 0 failed)...
  Processed 150